# Starbucks Rewards - EDA inicial

Objetivo desta etapa: ler os JSONs em `data/raw` e validar estrutura mínima antes de qualquer análise preditiva ou causal.

Escopo desta EDA:
- shape, colunas, tipos e primeiras linhas;
- nulos e duplicados;
- eventos do `transcript`;
- inspeção do campo aninhado `value`.

Não há modelagem nesta etapa.

## 1. Setup e leitura dos dados

Os arquivos são JSON Lines: cada linha contém um registro JSON independente.

In [5]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

files = {
    "portfolio": RAW_DIR / "portfolio.json",
    "profile": RAW_DIR / "profile.json",
    "transcript": RAW_DIR / "transcript.json",
}

for name, path in files.items():
    print(f"{name}: {path} | exists={path.exists()} | size_mb={path.stat().st_size / 1024**2:.2f}")

portfolio: c:\GitHub\datascience\projetos\starbucks-customer-rewards-program-dataset\data\raw\portfolio.json | exists=True | size_mb=0.00
profile: c:\GitHub\datascience\projetos\starbucks-customer-rewards-program-dataset\data\raw\profile.json | exists=True | size_mb=1.93
transcript: c:\GitHub\datascience\projetos\starbucks-customer-rewards-program-dataset\data\raw\transcript.json | exists=True | size_mb=38.67


In [6]:
dfs = {
    name: pd.read_json(path, lines=True)
    for name, path in files.items()
}

portfolio = dfs["portfolio"]
profile = dfs["profile"]
transcript = dfs["transcript"]

## 2. Shape, colunas, tipos e head

In [7]:
for name, df in dfs.items():
    print("=" * 80)
    print(name.upper())
    print(f"shape: {df.shape}")
    print("\ncolunas:")
    print(list(df.columns))
    print("\ntipos:")
    print(df.dtypes)
    display(df.head())

PORTFOLIO
shape: (10, 6)

colunas:
['reward', 'channels', 'difficulty', 'duration', 'offer_type', 'id']

tipos:
reward         int64
channels      object
difficulty     int64
duration       int64
offer_type       str
id               str
dtype: object


,reward,channels,difficulty,duration,offer_type,id
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"[web, email, mobile]",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"[web, email]",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


PROFILE
shape: (17000, 5)

colunas:
['gender', 'age', 'id', 'became_member_on', 'income']

tipos:
gender                  str
age                   int64
id                      str
became_member_on      int64
income              float64
dtype: object


,gender,age,id,became_member_on,income
0,NaN,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,NaN,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN
3,F,75,78afa995795e4d85b5d9ceeca43f5fef,20170509,100000.0
4,NaN,118,a03223e636434f42ac4c3df47e8bac43,20170804,NaN


TRANSCRIPT
shape: (306534, 4)

colunas:
['person', 'event', 'value', 'time']

tipos:
person       str
event        str
value     object
time       int64
dtype: object


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


## 3. Nulos e duplicados

Duplicados abaixo consideram a linha inteira. Unicidade de chaves será validada em etapa posterior, depois de confirmar a granularidade esperada.

In [8]:
import json


def make_hashable_for_duplicate_check(value):
    if isinstance(value, (dict, list)):
        return json.dumps(value, sort_keys=True)
    return value


def count_full_row_duplicates(df):
    comparable_df = df.map(make_hashable_for_duplicate_check)
    return int(comparable_df.duplicated().sum())


quality_rows = []

for name, df in dfs.items():
    quality_rows.append(
        {
            "tabela": name,
            "linhas": len(df),
            "colunas": df.shape[1],
            "duplicados_linha_inteira": count_full_row_duplicates(df),
        }
    )

quality_summary = pd.DataFrame(quality_rows)
display(quality_summary)

,tabela,linhas,colunas,duplicados_linha_inteira
0,portfolio,10,6,0
1,profile,17000,5,0
2,transcript,306534,4,397


In [9]:
for name, df in dfs.items():
    nulls = (
        df.isna()
        .sum()
        .rename("nulos")
        .to_frame()
    )
    nulls["pct_nulos"] = (nulls["nulos"] / len(df)).round(4)

    print("=" * 80)
    print(name.upper())
    display(nulls.sort_values("nulos", ascending=False))

PORTFOLIO


,nulos,pct_nulos
reward,0,0.0
channels,0,0.0
difficulty,0,0.0
duration,0,0.0
offer_type,0,0.0
id,0,0.0


PROFILE


,nulos,pct_nulos
gender,2175,0.1279
income,2175,0.1279
age,0,0.0000
id,0,0.0000
became_member_on,0,0.0000


TRANSCRIPT


,nulos,pct_nulos
person,0,0.0
event,0,0.0
value,0,0.0
time,0,0.0


## 4. Eventos do transcript

Esta etapa apenas descreve os eventos disponíveis e sua distribuição temporal. Ainda não atribui transações a ofertas nem define tratamento, controle ou outcome.

In [10]:
event_summary = (
    transcript["event"]
    .value_counts(dropna=False)
    .rename_axis("event")
    .reset_index(name="registros")
)
event_summary["pct"] = (event_summary["registros"] / len(transcript)).round(4)

display(event_summary)

,event,registros,pct
0,transaction,138953,0.4533
1,offer received,76277,0.2488
2,offer viewed,57725,0.1883
3,offer completed,33579,0.1095


In [11]:
time_summary = (
    transcript
    .groupby("event", dropna=False)["time"]
    .agg(registros="count", min_time="min", max_time="max", median_time="median")
    .reset_index()
    .sort_values("event")
)

display(time_summary)

,event,registros,min_time,max_time,median_time
0,offer completed,33579,0,714,432.0
1,offer received,76277,0,576,408.0
2,offer viewed,57725,0,714,408.0
3,transaction,138953,0,714,402.0


## 5. Inspeção do campo `value`

O campo `value` é aninhado e muda conforme o tipo de evento. Esta inspeção identifica suas chaves sem assumir ainda regra de atribuição entre oferta e transação.

In [12]:
def value_keys(value):
    if isinstance(value, dict):
        return tuple(sorted(value.keys()))
    return tuple()


value_key_summary = (
    transcript
    .assign(value_keys=transcript["value"].map(value_keys))
    .groupby(["event", "value_keys"], dropna=False)
    .size()
    .reset_index(name="registros")
    .sort_values(["event", "registros"], ascending=[True, False])
)

display(value_key_summary)

,event,value_keys,registros
0,offer completed,"(offer_id, reward)",33579
1,offer received,"(offer id,)",76277
2,offer viewed,"(offer id,)",57725
3,transaction,"(amount,)",138953


In [13]:
value_expanded = pd.json_normalize(transcript["value"])
value_expanded.columns = [col.replace(" ", "_") for col in value_expanded.columns]

transcript_value_preview = pd.concat(
    [transcript[["person", "event", "time"]], value_expanded],
    axis=1,
)

print("colunas extraidas de value:")
print(list(value_expanded.columns))

display(transcript_value_preview.head(10))

colunas extraidas de value:
['offer_id', 'amount', 'offer_id', 'reward']


,person,event,time,offer_id,amount,offer_id,reward
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,NaN
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,NaN
5,389bc3fa690240e798340f5a15918d5c,offer received,0,f19421c1d4aa40978ebb69ca19b0e20d,NaN,NaN,NaN
6,c4863c7985cf408faee930f111475da3,offer received,0,2298d6c36e964ae4a3e7e9706d1fb8c2,NaN,NaN,NaN
7,2eeac8d8feae4a8cad5a6af0499a211d,offer received,0,3f207df678b143eea3cee63160fa8bed,NaN,NaN,NaN
8,aa4862eba776480b8bb9c68455b8c2e1,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN
9,31dda685af34476cad5bc968bdb01c53,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,NaN


In [14]:
for event_name in transcript["event"].dropna().sort_values().unique():
    print("=" * 80)
    print(event_name)
    display(
        transcript.loc[transcript["event"] == event_name, ["person", "event", "value", "time"]]
        .head(5)
    )

offer completed


,person,event,value,time
12658,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,offer completed,"{'offer_id': '2906b810c7d4411798c6938adc9daaa5', 'reward': 2}",0
12672,fe97aa22dd3e48c8b143116a8403dd52,offer completed,"{'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4', 'reward': 2}",0
12679,629fc02d56414d91bca360decdfa9288,offer completed,"{'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9', 'reward': 5}",0
12692,676506bad68e4161b9bbaffeb039626b,offer completed,"{'offer_id': 'ae264e3637204a6fb9bb56bc8210ddfd', 'reward': 10}",0
12697,8f7dd3b2afe14c078eb4f6e6fe4ba97d,offer completed,"{'offer_id': '4d5c57ea9a6940dd891ad53e9dbe8da0', 'reward': 10}",0


offer received


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,68617ca6246f4fbc85e91a2a49552598,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


offer viewed


,person,event,value,time
12650,389bc3fa690240e798340f5a15918d5c,offer viewed,{'offer id': 'f19421c1d4aa40978ebb69ca19b0e20d'},0
12651,d1ede868e29245ea91818a903fec04c6,offer viewed,{'offer id': '5a8bc65990b245e5a138643cd4eb9837'},0
12652,102e9454054946fda62242d2e176fdce,offer viewed,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0
12653,02c083884c7d45b39cc68e1314fec56c,offer viewed,{'offer id': 'ae264e3637204a6fb9bb56bc8210ddfd'},0
12655,be8a5d1981a2458d90b255ddc7e0d174,offer viewed,{'offer id': '5a8bc65990b245e5a138643cd4eb9837'},0


transaction


,person,event,value,time
12654,02c083884c7d45b39cc68e1314fec56c,transaction,{'amount': 0.8300000000000001},0
12657,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,transaction,{'amount': 34.56},0
12659,54890f68699049c2a04d415abc25e717,transaction,{'amount': 13.23},0
12670,b2f1cd155b864803ad8334cdf13c4bd2,transaction,{'amount': 19.51},0
12671,fe97aa22dd3e48c8b143116a8403dd52,transaction,{'amount': 18.97},0


## 6. Observações para validação manual antes de avançar

- Confirmar a unidade do campo `time` e se ela é compatível com `duration` em `portfolio`.
- Confirmar se `profile.age == 118` representa idade real ou valor sentinela para cadastro incompleto.
- Confirmar se `value.offer id` e `value.offer_id` representam a mesma chave de oferta em eventos diferentes.
- Confirmar se existe grupo controle real ou se a ausência de oferta será apenas comparação observacional.
- Não definir tratamento, controle, outcome ou janela causal antes dessas validações.